# Psoriasis scRNA-seq Drug Repurposing Pipeline

**Glen Ritschel | Ritschel Research | 2026**

GitHub: https://github.com/glenritschel/psoriasis-scrna

**Dataset**: GSE173706 — PP (lesional) vs PN (uninvolved psoriatic), 25 samples, CSV/Ensembl format.

**Strategy**: Google Drive is mounted first. All intermediate files save directly to Drive after each step. If the runtime resets, rerun from the failed step — completed steps reload from Drive automatically.

---
**Before running**: Runtime > Change runtime type > **T4 GPU**

## Step 1 — Mount Google Drive
Run this first. All outputs write directly to Drive throughout the pipeline.

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Drive mounted.")
print("Output directory:", DRIVE_DIR)
print("Existing files:", sorted(os.listdir(DRIVE_DIR)))

## Step 2 — Clone repo and install dependencies

In [ ]:
import os
if not os.path.exists("/content/psoriasis-scrna"):
    !git clone https://github.com/glenritschel/psoriasis-scrna.git
else:
    !cd /content/psoriasis-scrna && git pull
%cd /content/psoriasis-scrna/notebooks
print("Working directory:", os.getcwd())

In [ ]:
import subprocess, sys
packages = [
    "scvi-tools==1.3.3", "scanpy==1.11.5", "gseapy==1.1.10",
    "leidenalg", "python-igraph", "scikit-misc", "biopython", "pybiomart"
]
for pkg in packages:
    print("Installing", pkg, "...")
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)
print("All dependencies installed.")

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("WARNING: No GPU. Runtime > Change runtime type > T4 GPU")

## Step 3 — Download raw data to Drive

Downloads GSE173706_RAW.tar (252 MB) and the Ensembl ID → symbol mapping.
Both are saved to Drive and reused on subsequent runs automatically.

In [ ]:
import os, urllib.request, tarfile, glob

DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
RAW_DIR = os.path.join(DRIVE_DIR, "raw")
os.makedirs(RAW_DIR, exist_ok=True)

# Download tar
TAR_URL = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE173nnn/GSE173706/suppl/GSE173706_RAW.tar"
TAR_PATH = os.path.join(RAW_DIR, "GSE173706_RAW.tar")
if not os.path.exists(TAR_PATH):
    print("Downloading GSE173706_RAW.tar (252 MB) to Drive...", flush=True)
    urllib.request.urlretrieve(TAR_URL, TAR_PATH)
    print("Done.", round(os.path.getsize(TAR_PATH) / 1e6), "MB")
else:
    print("TAR already on Drive.")

# Extract
if len(glob.glob(os.path.join(RAW_DIR, "*.csv.gz"))) < 10:
    print("Extracting...")
    with tarfile.open(TAR_PATH) as tf:
        tf.extractall(RAW_DIR)
    print("Extraction complete.")

csv_files = sorted(glob.glob(os.path.join(RAW_DIR, "GSM*.csv.gz")))
print(f"CSV files ready: {len(csv_files)}")
for cond in ["PP", "PN", "NS"]:
    n = sum(1 for f in csv_files if f"_{cond}-" in os.path.basename(f))
    print(f"  {cond}: {n} samples")

In [ ]:
import os, pandas as pd
DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
RAW_DIR = os.path.join(DRIVE_DIR, "raw")
MAPPING_PATH = os.path.join(RAW_DIR, "ensembl_to_symbol.csv")

if not os.path.exists(MAPPING_PATH):
    from pybiomart import Dataset
    print("Querying Ensembl BioMart (~1 min)...")
    ds = Dataset(name="hsapiens_gene_ensembl", host="http://www.ensembl.org")
    mapping = ds.query(attributes=["ensembl_gene_id", "hgnc_symbol"])
    mapping.to_csv(MAPPING_PATH, index=False)
    print("Saved:", len(mapping), "entries ->", MAPPING_PATH)
else:
    mapping = pd.read_csv(MAPPING_PATH)
    print("Mapping already on Drive:", len(mapping), "entries")
print(mapping.head())

## Step 4 — Load, QC, and save to Drive

Loads PP and PN samples one condition at a time to manage memory.
Each batch saves to Drive before the next is loaded.
**If this step completed in a previous run, it will reload from Drive and skip loading.**

In [ ]:
import os, re, glob, gc
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
RAW_DIR   = os.path.join(DRIVE_DIR, "raw")
MAPPING_PATH = os.path.join(RAW_DIR, "ensembl_to_symbol.csv")
QC_PATH   = os.path.join(DRIVE_DIR, "adata_qc.h5ad")

QC_MIN_GENES   = 200
QC_MIN_CELLS   = 3
QC_MAX_MITO    = 20.0
N_HVG          = 4000
USE_CONDITIONS = ["PP", "PN"]

if os.path.exists(QC_PATH):
    print("QC object already on Drive. Loading...")
    adata_qc = sc.read_h5ad(QC_PATH)
    print("Loaded:", adata_qc.n_obs, "cells x", adata_qc.n_vars, "genes")
    for c in USE_CONDITIONS:
        print(f"  {c}:", (adata_qc.obs["condition"] == c).sum())
else:
    # Load Ensembl mapping
    mapping_df = pd.read_csv(MAPPING_PATH)
    ensg_col = [c for c in mapping_df.columns if "stable" in c.lower() or "ensembl" in c.lower()][0]
    sym_col  = [c for c in mapping_df.columns if "symbol" in c.lower() or "hgnc" in c.lower()][0]
    ensg_to_symbol = {k: v for k, v in zip(mapping_df[ensg_col], mapping_df[sym_col])
                      if isinstance(v, str) and v.strip() != ""}
    del mapping_df; gc.collect()
    print("Mapping loaded:", len(ensg_to_symbol), "entries")

    def parse_filename(fname):
        m = re.match(r"(GSM\d+)_(NS|PN|PP)-(.+)\.csv\.gz$", fname)
        return (m.group(1), m.group(2), m.group(3)) if m else (None, None, None)

    def load_csv_sample(csv_path, sample_id, condition, patient):
        df = pd.read_csv(csv_path, index_col=0).T
        df.columns = df.columns.map(lambda g: ensg_to_symbol.get(g, g))
        df = df.loc[:, df.columns != ""]
        df = df.loc[:, ~df.columns.duplicated()]
        adata = sc.AnnData(X=sp.csr_matrix(df.values, dtype=np.float32))
        adata.obs_names = [sample_id + "_" + bc for bc in df.index]
        adata.var_names = df.columns.tolist()
        adata.var_names_make_unique()
        del df; gc.collect()
        sc.pp.filter_cells(adata, min_counts=1)
        adata.obs["sample"]    = sample_id
        adata.obs["condition"] = condition
        adata.obs["patient"]   = patient
        return adata

    csv_files = sorted(glob.glob(os.path.join(RAW_DIR, "GSM*.csv.gz")))
    batch_paths = []

    for cond in USE_CONDITIONS:
        batch_path = os.path.join(DRIVE_DIR, f"batch_{cond}.h5ad")
        if os.path.exists(batch_path):
            print(f"Batch {cond} already on Drive, skipping.")
            batch_paths.append(batch_path)
            continue
        print(f"\nLoading {cond} samples...")
        adatas = []
        for f in csv_files:
            gsm, condition, patient = parse_filename(os.path.basename(f))
            if condition != cond:
                continue
            sid = gsm + "_" + cond + "_" + patient
            print(f"  {sid}...", end=" ", flush=True)
            a = load_csv_sample(f, sid, cond, patient)
            adatas.append(a)
            print(a.n_obs, "cells")
        print(f"  Concatenating {cond}...")
        batch = sc.concat(adatas, join="outer", fill_value=0)
        del adatas; gc.collect()
        print(f"  {cond}: {batch.n_obs} cells — saving to Drive...")
        batch.write_h5ad(batch_path)
        del batch; gc.collect()
        print(f"  Saved: {batch_path}")
        batch_paths.append(batch_path)

    # Concatenate both batches
    print("\nConcatenating batches from Drive...")
    adatas = [sc.read_h5ad(p) for p in batch_paths]
    adata = sc.concat(adatas, join="outer", fill_value=0)
    del adatas; gc.collect()
    adata.layers["counts"] = sp.csr_matrix(adata.X) if not sp.issparse(adata.X) else adata.X.copy()
    print("Combined:", adata.n_obs, "cells x", adata.n_vars, "genes")

    # QC
    print("\nRunning QC...")
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
    n_before = adata.n_obs
    sc.pp.filter_cells(adata, min_genes=QC_MIN_GENES)
    sc.pp.filter_genes(adata, min_cells=QC_MIN_CELLS)
    adata = adata[adata.obs["pct_counts_mt"] < QC_MAX_MITO].copy()
    print(f"QC: {n_before} -> {adata.n_obs} cells")

    # HVG
    print("Selecting HVGs...")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.layers["norm_log"] = adata.X.copy()
    sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor="seurat_v3",
                                layer="counts", batch_key="sample", subset=False)
    print("HVGs:", adata.var["highly_variable"].sum())

    # Save to Drive
    print("Saving QC object to Drive...")
    adata.write_h5ad(QC_PATH)
    adata_qc = adata
    print("Saved:", QC_PATH)
    for c in USE_CONDITIONS:
        print(f"  {c}:", (adata_qc.obs["condition"] == c).sum())

print("\nStep 4 complete.")

## Step 5 — scVI Embedding + Leiden Clustering

Trains scVI on the HVG subset with GPU acceleration.
Saves scVI object to Drive on completion.

In [ ]:
import os, gc
import numpy as np
import scanpy as sc
import scvi

DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
QC_PATH   = os.path.join(DRIVE_DIR, "adata_qc.h5ad")
SCVI_PATH = os.path.join(DRIVE_DIR, "adata_scvi.h5ad")

LEIDEN_RESOLUTIONS = [0.5, 0.8, 1.2]
RANDOM_SEED = 0

if os.path.exists(SCVI_PATH):
    print("scVI object already on Drive. Loading...")
    adata_scvi = sc.read_h5ad(SCVI_PATH)
    print("Loaded:", adata_scvi.n_obs, "cells,",
          adata_scvi.obs["leiden"].nunique(), "clusters")
else:
    if "adata_qc" not in dir():
        print("Loading QC object from Drive...")
        adata_qc = sc.read_h5ad(QC_PATH)

    adata_hvg = adata_qc[:, adata_qc.var["highly_variable"]].copy()
    print("HVG subset:", adata_hvg.n_obs, "x", adata_hvg.n_vars)

    # Train scVI
    import torch, random
    np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
    scvi.settings.seed = RANDOM_SEED
    scvi.model.SCVI.setup_anndata(adata_hvg, layer="counts", batch_key="sample")
    model = scvi.model.SCVI(adata_hvg, n_latent=30, n_layers=2, n_hidden=128)
    print("Training scVI on", adata_hvg.n_obs, "cells...")
    model.train(max_epochs=400, early_stopping=False, accelerator="auto")
    final_loss = float(np.array(model.history["train_loss_epoch"].values[-1]).flat[0])
    print("Training complete. Final loss:", round(final_loss, 2))
    adata_hvg.obsm["X_scVI"] = model.get_latent_representation()

    # Neighbour graph and UMAP
    sc.pp.neighbors(adata_hvg, use_rep="X_scVI", n_neighbors=15)
    sc.tl.umap(adata_hvg)

    # Leiden at multiple resolutions
    for res in LEIDEN_RESOLUTIONS:
        key = f"leiden_{res}"
        sc.tl.leiden(adata_hvg, resolution=res, key_added=key,
                     random_state=RANDOM_SEED, flavor="igraph",
                     n_iterations=2, directed=False)
        print(f"  Resolution {res}: {adata_hvg.obs[key].nunique()} clusters")

    # Default to resolution 0.8; adjust after reviewing resolution_metrics
    adata_hvg.obs["leiden"] = adata_hvg.obs["leiden_0.8"].copy()
    print("Using resolution 0.8:", adata_hvg.obs["leiden"].nunique(), "clusters")

    print("Saving scVI object to Drive...")
    adata_hvg.write_h5ad(SCVI_PATH)
    adata_scvi = adata_hvg
    print("Saved:", SCVI_PATH)

print("\nStep 5 complete.")

## Step 6 — Copy scVI object locally for scripts 03–07

Scripts 03–07 read and write `data/processed/`. This cell copies the scVI
object from Drive to the local path so the scripts can find it.

In [ ]:
import os, shutil

DRIVE_DIR      = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
LOCAL_PROCESSED = "/content/psoriasis-scrna/data/processed"
os.makedirs(LOCAL_PROCESSED, exist_ok=True)

for fname in ["adata_scvi.h5ad", "adata_qc.h5ad"]:
    src = os.path.join(DRIVE_DIR, fname)
    dst = os.path.join(LOCAL_PROCESSED, fname)
    if os.path.exists(src) and not os.path.exists(dst):
        print(f"Copying {fname} from Drive...", end=" ", flush=True)
        shutil.copy2(src, dst)
        print(f"{os.path.getsize(dst)//1e6:.0f} MB")
    elif os.path.exists(dst):
        print(f"Already local: {fname}")
    else:
        print(f"NOT FOUND on Drive: {fname}")

print("Local processed dir:", os.listdir(LOCAL_PROCESSED))

## Step 7 — Script 03: Cell Type Annotation

In [ ]:
import os; os.chdir("/content/psoriasis-scrna/notebooks")
%run ../src/03_annotate_clusters.py

In [ ]:
# Save annotated object to Drive immediately
import shutil, os
DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
for fname in ["adata_annotated.h5ad", "cluster_annotations.csv", "cluster_marker_scores.csv"]:
    src = f"/content/psoriasis-scrna/data/processed/{fname}"
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_DIR, fname))
        print("Saved:", fname)

## Step 8 — Script 04: Psoriasis Signature Scoring

In [ ]:
import os; os.chdir("/content/psoriasis-scrna/notebooks")
%run ../src/04_signature_scoring.py

In [ ]:
import shutil, os, glob
DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
for f in glob.glob("/content/psoriasis-scrna/data/processed/signature_scores*.csv") + \
         ["/content/psoriasis-scrna/data/processed/adata_scored.h5ad"]:
    if os.path.exists(f):
        shutil.copy2(f, os.path.join(DRIVE_DIR, os.path.basename(f)))
        print("Saved:", os.path.basename(f))

## Step 9 — Script 05: Differential Expression (PP vs PN)

In [ ]:
import os; os.chdir("/content/psoriasis-scrna/notebooks")
%run ../src/05_differential_expression.py

In [ ]:
import shutil, os, glob
DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
for f in glob.glob("/content/psoriasis-scrna/data/processed/de_*.csv") + \
         ["/content/psoriasis-scrna/data/processed/adata_de.h5ad"]:
    if os.path.exists(f):
        shutil.copy2(f, os.path.join(DRIVE_DIR, os.path.basename(f)))
        print("Saved:", os.path.basename(f))

## Step 10 — Script 06: LINCS L1000 Reversal Scoring

This step takes 30–60 minutes. Do not close the browser tab.

In [ ]:
import os; os.chdir("/content/psoriasis-scrna/notebooks")
%run ../src/06_lincs_repurposing.py

In [ ]:
# Save LINCS results to Drive immediately after script 06 completes
import shutil, os, glob
DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
for f in glob.glob("/content/psoriasis-scrna/data/processed/lincs_*.csv"):
    shutil.copy2(f, os.path.join(DRIVE_DIR, os.path.basename(f)))
    print("Saved:", os.path.basename(f))

## Step 11 — Script 07: Novelty Assessment + Priority Scoring

In [ ]:
import os; os.chdir("/content/psoriasis-scrna/notebooks")
%run ../src/07_novelty_prioritization.py

In [ ]:
# Save all remaining outputs to Drive
import shutil, os, glob
DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
for f in glob.glob("/content/psoriasis-scrna/data/processed/*.csv"):
    dst = os.path.join(DRIVE_DIR, os.path.basename(f))
    shutil.copy2(f, dst)
    print("Saved:", os.path.basename(f))
print("\nPipeline complete. All outputs on Drive.")

## Step 12 — Verify outputs on Drive

In [ ]:
import os, glob
DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
files = sorted(
    glob.glob(os.path.join(DRIVE_DIR, "*.csv")) +
    glob.glob(os.path.join(DRIVE_DIR, "*.h5ad"))
)
print(f"{len(files)} output files on Drive:")
for f in files:
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {os.path.basename(f)} ({size_mb:.0f} MB)")